<a href="https://colab.research.google.com/github/balajiduddukuri/Langchain_practice/blob/Ultimate-Content-Repurposer/RAG_MAR_14th.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install openai langchain chromadb faiss-cpu pypdf tiktoken docarray langchain_openai

In [ ]:
pip show langchain

In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get("api_key")

print("OPENAI API key is loaded successfully")

- **Week 9 Agenda:**

- Advanced LangChain Concepts & Applications Deep Dive into LangChain Models & Prompts

- Review of LLMs and Chat Models (more on parameters, temperature, token limits)
Advanced Prompt Engineering: Few-shot prompting, prompt chaining, dynamic prompt creation
- Prompt Templates: Customizing and optimizing ChatPromptTemplate for specific use cases
- Understanding and Building Chains

- **Introduction to Chains:** Combining LLMs with other components
Types of Chains: LLMChain, SimpleSequentialChain, RunnableSequence
Building multi-step workflows with LCEL for complex tasks
Retrieval Augmented Generation (RAG) Fundamentals

- **Document Loaders:** Loading data from various sources (PDFs, web pages, databases)
- **Text Splitters:** Breaking down large documents into manageable chunks
Embeddings: Converting text into numerical representations for similarity search
Vector Stores: Storing and querying embedded documents (e.g., ChromaDB, FAISS)
Introduction to LangChain Agents

**What are Agents? Dynamic decision-making with LLMs**
- Tools: Giving agents access to external capabilities (e.g., search, calculators)
Simple Agent Creation: Designing agents to solve problems by choosing actions dynamically

In [41]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = "gpt-4o-mini")

In [ ]:
llm = ChatOpenAI(
                  model = "gpt-4o-mini",
                  max_tokens =200, # limits response length
                  top_p = 0.8, # controls probability
                  temperature = 0 # now u get till 2 -more number more randomness -> higher more random ness - low-0, Medium -0.5, High 1
                )

In [42]:
#Fewshot prompting in LangChain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment analysis assistant."),

    ("human", "Text: I love this product"),
    ("ai", "Sentiment: Positive"),

    ("human", "Text: This is the worst service ever."),
    ("ai", "Sentiment: Negative"),

    ("human", "Text: The delivery was late but the product is good.")
])

chain = prompt | llm

result = chain.invoke({})

print(result.content)


Sentiment: Mixed (Negative for delivery, Positive for product)


In [47]:
#Runnable Sequence
#Input -> Prompt -> LLM -> Output Input -> Prompt 1 -> LLm -> P2 -> LLM -> Final Output
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Explain {topic} in simple terms"
)

# Load LLM:
llm = ChatOpenAI(model = "gpt-4o-mini")

# Create Runnable sequences:
from langchain_core.runnables import RunnableSequence
chain = RunnableSequence(prompt, llm)

# This creates a 2-step pipeline:
## Input -> PromptTemplate -> LLM genrating ans -> Out

# Run the Chain:
response = chain.invoke({"topic": "super heros"})
print(response.content)

Superheroes are fictional characters who usually have special powers or abilities that make them different from ordinary people. They often fight against bad guys, protect the innocent, and try to make the world a better place. Superheroes can fly, have super strength, or even the ability to become invisible. They often wear distinctive costumes, have secret identities, and can be found in comic books, movies, and TV shows. Some famous superheroes include Superman, Spider-Man, and Wonder Woman.


In [50]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import PromptTemplate

def add_one(x: int) -> int:
    return x + 1

def mul_two(x: int) -> int:
    return x * 2

runnable_1 = RunnableLambda(add_one)
runnable_2 = RunnableLambda(mul_two)
sequence = runnable_1 | runnable_2
# Or equivalently:
# sequence = RunnableSequence(first=runnable_1, last=runnable_2)
sequence.invoke(1)
# await sequence.ainvoke(1) # ainvoke requires an async context

sequence.batch([1, 2, 3])
# await sequence.abatch([1, 2, 3]) # abatch requires an async context

# Here's an example that uses streams JSON output generated by an LLM:

from langchain_core.output_parsers.json import SimpleJsonOutputParser
from langchain_openai import ChatOpenAI

prompt = PromptTemplate.from_template(
    "In JSON format, give me a list of {topic} and their "
    "corresponding names in French, Spanish and in a "
    "Cat Language."
)

model = ChatOpenAI()
chain = prompt | model | SimpleJsonOutputParser()

# The astream call also requires an async context, which is not directly available in a standard Colab cell execution.
# To demonstrate, we can run a single sync invoke instead or set up an async event loop.
# For simplicity, I'm commenting out the async stream and batch calls for now.
# async for chunk in chain.astream({"topic": "colors"}):
#     print("-")  # noqa: T201
#     print(chunk, sep="", flush=True)  # noqa: T201

# Example of synchronous call for JSON output:
result_json = chain.invoke({"topic": "colors"})
print(result_json)

{'colors': [{'name': 'blue', 'french': 'bleu', 'spanish': 'azul', 'cat_language': 'meow'}, {'name': 'green', 'french': 'vert', 'spanish': 'verde', 'cat_language': 'purrr'}, {'name': 'red', 'french': 'rouge', 'spanish': 'rojo', 'cat_language': 'purrpurr'}, {'name': 'yellow', 'french': 'jaune', 'spanish': 'amarillo', 'cat_language': 'meowrrow'}, {'name': 'purple', 'french': 'violet', 'spanish': 'morado', 'cat_language': 'purrrpurrr'}]}


**RAG**
Simple SequenceChains are no replaced in the latest langchain versions by LCEL

Retrieval Augmented Generation (RAG) Fundamentals

Document Loaders: Loading data from various sources (PDFs, web pages, databases) Text Splitters: Breaking down large documents into manageable chunks Embeddings: Converting text into numerical representations for similarity search Vector Stores: Storing and querying embedded documents (e.g., ChromaDB, FAISS)

LLMs like GPT are trained on large datasets they have limitations:
LLMs cannot access:
Private company documents
Latest information
Internal Databases
PDFs that is uploaded by users cannot be accessed
THE LLM does not know your company informations because the LLM was not trained on that document.

RAG -> Retrieval Augmented Generation

Retrieve relevent documents + Generate answer using LLM

RAG Pipeline:

User Question -> Retriever (Vector Search) -> Relevant Documents -> Prompt + Context -> LLM -> Answer

RAG allows the LLM to answer questions using external knowledge sources.

Core components of RAG: Document Loader -> Text Spllitter -> Embeddings -> Vector Store
Document Loader(DL)

DL are responsible for loading data into LangChain. Data Sources can include:

PDFs
Websites
text files
databases
CSV file
APIs

In [55]:
'''
Document(
    page_content="transformers use attention mechanisms...",
    metadata = {"source": "Mirdhaad-Book.pdf"}
)
'''

'\nDocument(\n    page_content="transformers use attention mechanisms...",\n    metadata = {"source": "Mirdhaad-Book.pdf"}\n)\n'

In [52]:
!pip -q install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible

In [68]:
!pip install --upgrade -q langchain langchain-community langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 8.5 MB/s eta 0:00:00


In [54]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Mirdhaad-Book.pdf")

documents = loader.load()

print(documents[0])

page_content='1
 
 
     T H E  
BOOK OF 
MIRDAD 
 
 
 
 
 
THE STRANGE STORY OF A MONASTERY 
WHICH WAS ONCE CALLED THE ARK 
 
 
 
 
 
 
MIKHAIL NAIMY' metadata={'producer': 'Acrobat Distiller 5.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2006-04-11T15:01:31+05:30', 'moddate': '2006-04-11T15:01:31+05:30', 'author': 'Administrator', 'title': 'Microsoft Word - The Book of Mirdad.doc', 'source': 'Mirdhaad-Book.pdf', 'total_pages': 140, 'page': 0, 'page_label': '1'}


In [70]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
# Removed: from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [71]:
# ------------------------------
# 1. Document Loader (DL)
# ------------------------------
pdf_path = "Mirdhaad-Book.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Loaded {len(documents)} pages")

# ------------------------------
# 2. Text Splitter
# ------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print(f"Split into {len(docs)} chunks")

Loaded 140 pages
Split into 450 chunks


In [73]:
# ------------------------------
# 3. Embeddings
# ------------------------------
embeddings = OpenAIEmbeddings(
    openai_api_key= userdata.get("api_key")
)

# ------------------------------
# 4. Vector Store
# ------------------------------
vectorstore = FAISS.from_documents(
    docs,
    embeddings
)

In [75]:
# ------------------------------
# 5. Retriever
# ------------------------------
retriever = vectorstore.as_retriever()

# ------------------------------
# 6. LLM
# ------------------------------
llm = ChatOpenAI(
    temperature=0,
    model="gpt-4o-mini"
    )

In [78]:
# ------------------------------
# 7. RAG QA Chain using LCEL
# ------------------------------

# Create a prompt template
rag_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know."),
    ("human", "{question}\nContext: {context}")
])

# Define the RAG chain
qa_chain = (
    {
        "context": (lambda x: x["question"]) | retriever | RunnableLambda(lambda docs: "\n\n".join(doc.page_content for doc in docs)),
        "question": (lambda x: x["question"])
    }
    | rag_prompt_template
    | llm
    | StrOutputParser()
)

print("RAG QA Chain created successfully using LCEL.")

RAG QA Chain created successfully using LCEL.


In [80]:
# ------------------------------
# 8. Query the PDF
# ------------------------------
query = "Summarize the main topic of this document in 10 lines"

response = qa_chain.invoke({"question": query})

print("\nAnswer:")
print(response)


Answer:
The document explores the profound connection between the heart, mind, and will in the pursuit of desires and fulfillment in life. It emphasizes the heart as the central hub of emotions, desires, and fears, while the mind serves as a disciplinarian and the will as a commander. The text suggests that achieving a singular, dominant desire can lead to its fulfillment, highlighting the importance of focus and clarity in one's aspirations. The narrative includes a metaphorical journey where a man, after enduring hardship, encounters a vision of hope and rejuvenation represented by a beautiful maiden. This encounter symbolizes the potential for renewal and the realization of dreams. Additionally, the document reflects on the nature of communication, suggesting that excessive words can obscure true meaning, and questions the efficacy of prayer, implying that often our requests go unanswered. Overall, it conveys a message about the significance of inner strength, clarity of purpose, a

**SAMPLE 2:** SARVESH

In [83]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Mirdhaad-Book.pdf")

documents = loader.load()

print(documents[0].page_content)

1
 
 
     T H E  
BOOK OF 
MIRDAD 
 
 
 
 
 
THE STRANGE STORY OF A MONASTERY 
WHICH WAS ONCE CALLED THE ARK 
 
 
 
 
 
 
MIKHAIL NAIMY


In [84]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,    # Chunk size = Size of chunk
    chunk_overlap = 50   # Overlap between chunks to preserve the context
)

chunks = splitter.split_documents(documents)

In [85]:
print(len(chunks))

779


In [86]:
for i, chunk in enumerate(chunks):
  print(f"Chunk {i+1}")
  print("Length:", len(chunk.page_content))
  print(chunk.page_content)

Chunk 1
Length: 136
1
 
 
     T H E  
BOOK OF 
MIRDAD 
 
 
 
 
 
THE STRANGE STORY OF A MONASTERY 
WHICH WAS ONCE CALLED THE ARK 
 
 
 
 
 
 
MIKHAIL NAIMY
Chunk 2
Length: 413
2
 
Table of contents 
 
 
 
THE BOUND ABBOTT............................................................................................................5 
FLINT SLOPE ...........................................................................................................................8 
THE KEEPER OF THE BOOK ..............................................................................................14
Chunk 3
Length: 497
THE BOOK .................................................................................................................................21 
CHAPTER ONE ..........................................................................................................................23 
MIRDAD UNVEILS HIMSELF AND SPEAKS ON VEILS AND SEALS..........................23 
CHAPTER TWO ...............

In [87]:
for i, chunk in enumerate(chunks):
  print(f"Chunk {i+1} size: {len(chunk.page_content)} characters")

Chunk 1 size: 136 characters
Chunk 2 size: 413 characters
Chunk 3 size: 497 characters
Chunk 4 size: 439 characters
Chunk 5 size: 471 characters
Chunk 6 size: 475 characters
Chunk 7 size: 395 characters
Chunk 8 size: 373 characters
Chunk 9 size: 415 characters
Chunk 10 size: 487 characters
Chunk 11 size: 455 characters
Chunk 12 size: 169 characters
Chunk 13 size: 454 characters
Chunk 14 size: 418 characters
Chunk 15 size: 428 characters
Chunk 16 size: 431 characters
Chunk 17 size: 482 characters
Chunk 18 size: 429 characters
Chunk 19 size: 403 characters
Chunk 20 size: 467 characters
Chunk 21 size: 447 characters
Chunk 22 size: 398 characters
Chunk 23 size: 467 characters
Chunk 24 size: 485 characters
Chunk 25 size: 427 characters
Chunk 26 size: 385 characters
Chunk 27 size: 441 characters
Chunk 28 size: 409 characters
Chunk 29 size: 412 characters
Chunk 30 size: 480 characters
Chunk 31 size: 410 characters
Chunk 32 size: 466 characters
Chunk 33 size: 243 characters
Chunk 34 size: 427 